In [9]:
from model_ranking.utils import load_h5
from model_ranking.consistency import calc_consistency_score

In [10]:
from typing import Any
from numpy.typing import NDArray
import numpy as np
from typing import Optional, Tuple, Literal
from scipy.spatial.distance import hamming

from model_ranking.metrics import calculate_EI_binary
from model_ranking.consistency import get_mask

def calc_seg_consistency_metric(
    aug_pred: NDArray[Any],
    no_aug_pred: NDArray[Any],
    metric: Literal[
        "EI",
        "Hamming-Distance",
    ],
    ignore_mask: Optional[NDArray[Any]] = None,
    threshold: float = 0.5,
) -> Tuple[NDArray[Any], NDArray[Any]]:
    """Calculate consistency metric between single aug pred array and non augmented pred array.
    The metric is calculated for pixels above threshold and set to None for non selected pixels.
    The output has the same shape as the input arrays with singleton dimensions removed.

    Args:
        aug_pred (np.ndarray): test time augmented prediction array
        no_aug_pred (np.ndarray): non augmented prediction array
        metric (Literal["Diff";, "EI", "Entropy"]): consistency metric type
        threshold (float, optional): prediction threshold to calculate consistency on. Defaults to 0.5.

    Returns:
        np.ndarray: consistency metric array with same shape as input arrays with singleton
        dimensions removed.
    """
    # Identify selected pixels (Union of aug and non
    # aug pixel predictions above threshold)
    mask = get_mask(no_aug_pred, aug_pred, threshold)
    if ignore_mask is not None:
        mask = np.logical_and(mask, ~ignore_mask.astype(bool))

    if metric == "EI":
        cmb_pred = np.stack([no_aug_pred, aug_pred], axis=0)
        hard_pred = cmb_pred > threshold
        metric_result, _, _, _ = calculate_EI_binary(hard_pred, cmb_pred)
    elif metric == "Hamming-Distance":
        if aug_pred.ndim == 2:
            if np.sum(mask) == 0:
                metric_result = np.array([np.nan])
            else:
                metric_result = np.array(
                    hamming(aug_pred[mask] > threshold, no_aug_pred[mask] > threshold)
                )
        else:
            metric_result = np.zeros(len(aug_pred))
            for i in range(len(aug_pred)):
                # if mask empty set to None
                if np.sum(mask[i]) == 0:
                    metric_result[i] = np.array([np.nan])
                else:
                    metric_result[i] = hamming(
                        (aug_pred[i][mask[i]] > threshold),
                        (no_aug_pred[i][mask[i]] > threshold),
                    )
        return metric_result, mask

    # Remove singleton dimensions
    metric_result = np.squeeze(metric_result)
    mask = np.squeeze(mask)
    # invert mask to get non selected pixels
    mask_inverted = np.logical_not(mask)
    # set non selected pixels to None
    metric_result[mask_inverted] = None
    return metric_result, mask

# Fullsize VNC

In [7]:
EtoV_DO02_path = "/g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/EPFL_to_VNC_gap/feature_perturbation_consistency/FP_3/E_model4/norm_Normalize/DO_a02/predictions/data_labeled_mito_DO_a02.h5"
EtoV_DO02_f1 = load_h5(EtoV_DO02_path, "hard_f1")
EtoV_DO02_EI = load_h5(EtoV_DO02_path, "EI_th05")
EtoV_DO02_HD = load_h5(EtoV_DO02_path, "HD_th05")
EtoV_DO02_pred = load_h5(EtoV_DO02_path, "predictions")

In [4]:
EtoV_none_path = "/g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/EPFL_to_VNC_gap/feature_perturbation_consistency/FP_3/E_model4/norm_Normalize/none/predictions/data_labeled_mito_none.h5"
EtoV_none_pred = load_h5(EtoV_none_path, "predictions")

In [15]:
from tqdm import tqdm

consis_scores_EI = np.zeros((len(EtoV_DO02_pred), 256, 256))
consis_scores_HD =np.zeros(len(EtoV_DO02_pred))
for i, (aug_pred, none_pred) in tqdm(enumerate((zip(EtoV_DO02_pred, EtoV_none_pred)))):
    consis_EI, _ = calc_seg_consistency_metric(
        aug_pred,
        none_pred,
        metric="EI",
    )
    consis_HD, _ = calc_seg_consistency_metric(
        aug_pred,
        none_pred,
        metric="Hamming-Distance",
    )
    consis_scores_EI[i] = consis_EI
    consis_scores_HD[i] = consis_HD

0it [00:00, ?it/s]/tmp/ipykernel_3630340/2907491267.py:17: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  consis_scores_HD[i] = consis_HD
/tmp/ipykernel_3630340/2532467924.py:57: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  metric_result[i] = np.array([np.nan])
320it [00:00, 646.28it/s]


In [16]:
# Compare EtoV_DO02_EI and consis_scores for equality or both being nan, up to a tolerance of 1e-6
equal_mask_EI = (
    (np.isnan(EtoV_DO02_EI) & np.isnan(consis_scores_EI)) |
    (np.isclose(EtoV_DO02_EI, consis_scores_EI, atol=1e-6, equal_nan=False))
)
equal_mask_HD = (
    (np.isnan(EtoV_DO02_HD) & np.isnan(consis_scores_HD)) |
    (np.isclose(EtoV_DO02_HD, consis_scores_HD, atol= 1e-6, equal_nan=False))
)
EI_all_equal = np.all(equal_mask_EI)
HD_all_equal = np.all(equal_mask_HD)

print("All entries equal or both nan for EI:", EI_all_equal)
print("All entries equal or both nan for HD:", HD_all_equal)

All entries equal or both nan for EI: True
All entries equal or both nan for HD: True


# Downsampled VNC

In [17]:
EtoV_DO02_path_downsampled = "/g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/EPFL_to_VNC_gap/feature_perturbation_consistency/resized/E_model4/norm_Normalize/DO_a02/predictions/source_mitoEM_true_predictions.h5"
EtoV_DO02_f1_downsampled = load_h5(EtoV_DO02_path_downsampled, "hard_f1")
EtoV_DO02_EI_downsampled = load_h5(EtoV_DO02_path_downsampled, "EI_consis")
EtoV_DO02_HD_downsampled = load_h5(EtoV_DO02_path_downsampled, "HD_consis")
EtoV_DO02_pred_downsampled = load_h5(EtoV_DO02_path_downsampled, "predictions")

In [18]:
EtoV_none_path_downsampled = "/g/kreshuk/talks/consistency_results/patch_segmentation/mitochondria/EPFL_to_VNC_gap/feature_perturbation_consistency/resized/E_model4/norm_Normalize/none/predictions/source_mitoEM_true_predictions.h5"
EtoV_none_pred_downsampled = load_h5(EtoV_none_path_downsampled, "predictions")

In [19]:
consis_scores_EI_downsampled = np.zeros((len(EtoV_DO02_pred_downsampled), 256, 256))
consis_scores_HD_downsampled =np.zeros(len(EtoV_DO02_pred_downsampled))
for i, (aug_pred, none_pred) in tqdm(enumerate((zip(EtoV_DO02_pred_downsampled, EtoV_none_pred_downsampled)))):
    consis_EI, _ = calc_seg_consistency_metric(
        aug_pred,
        none_pred,
        metric="EI",
    )
    consis_HD, _ = calc_seg_consistency_metric(
        aug_pred,
        none_pred,
        metric="Hamming-Distance",
    )
    consis_scores_EI_downsampled[i] = consis_EI
    consis_scores_HD_downsampled[i] = consis_HD

0it [00:00, ?it/s]/tmp/ipykernel_3630340/1535985158.py:15: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  consis_scores_HD_downsampled[i] = consis_HD
62it [00:00, 612.26it/s]/tmp/ipykernel_3630340/2532467924.py:57: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  metric_result[i] = np.array([np.nan])
180it [00:00, 624.18it/s]


In [20]:
# Compare EtoV_DO02_EI and consis_scores for equality or both being nan, up to a tolerance of 1e-6
equal_mask_EI = (
    (np.isnan(EtoV_DO02_EI_downsampled) & np.isnan(consis_scores_EI_downsampled)) |
    (np.isclose(EtoV_DO02_EI_downsampled, consis_scores_EI_downsampled, atol=1e-6, equal_nan=False))
)
equal_mask_HD = (
    (np.isnan(EtoV_DO02_HD_downsampled) & np.isnan(consis_scores_HD_downsampled)) |
    (np.isclose(EtoV_DO02_HD_downsampled, consis_scores_HD_downsampled, atol= 1e-6, equal_nan=False))
)
EI_all_equal = np.all(equal_mask_EI)
HD_all_equal = np.all(equal_mask_HD)

print("All entries equal or both nan for EI:", EI_all_equal)
print("All entries equal or both nan for HD:", HD_all_equal)

All entries equal or both nan for EI: True
All entries equal or both nan for HD: True
